# Final Assignment: training

This notebook contains the source code for training the weights of the classification model.

## Download dataset

The following cell needs to be ran only once. It downloads and unzips the data. The unzip CLI tool might not be installed by default on Windows, so if you are using windows, please unzip manually into a folder called "data", or see the README.md.

In [1]:
# Downloading the dataset (unzip might not be installed on Windows)
# !curl -L -o ./data.zip https://www.kaggle.com/api/v1/datasets/download/navoneel/brain-mri-images-for-brain-tumor-detection
# !unzip -d ./data ./data.zip

## Importing libraries

See README.md for installing dependencies :).

In [2]:
import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
from torchvision import models, datasets, transforms
from tqdm import tqdm

## Setting the seed

Setting the seed for good reproducability :D

In [3]:
torch.manual_seed(21)  # Using my lucky number :D
np.random.seed(21)

## Setting the device used for training

In [4]:
device = "cpu"
if torch.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
print(f"Using: {device}")

Using: cuda


## Preparing the dataset

In [5]:
transform = transforms.Compose(
    [
        transforms.Resize((299, 299)),  # 299 is the minimal image size of InceptionV3
        transforms.ToTensor()
    ]
)

In [6]:
path = os.path.join("data", "brain_tumor_dataset")

In [7]:
dataset = datasets.ImageFolder(
    root=path,
    transform=transform
)
train_dataset, test_dataset = random_split(
    dataset=dataset,
    lengths=[0.8, 0.2]
)

In [8]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

## Creating the model

### Loading pre-trained model

In [9]:
model = models.inception_v3(weights=models.Inception_V3_Weights)

C:\Users\daanw\PycharmProjects\SOW-BKI266\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


### Adding custom classification head

In [10]:
model.fc = nn.Linear(in_features=2048, out_features=2)  # Got the input features from the sourcecode hihi

In [11]:
model = model.to(device)

### Freezing layers

## Training the model

### Setting hyper parameters

In [12]:
EPOCHS = 100
LR = 1e-4

### Initializing optimizer and loss function

In [13]:
optimizer = Adam(params=model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss().to(device)

### Setting model to train mode and moving to device

In [14]:
model.train()

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

In [15]:
for epoch in range(EPOCHS):
    running_loss = 0

    for X, y in tqdm(train_loader, f"Epoch {epoch +1}"):
        X = X.to(device)
        y = y.to(device)

        y_pred = model(X)[0]
        loss = criterion(y_pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch complete: {epoch + 1}/{EPOCHS} (loss: {running_loss})")

Epoch 1: 100%|██████████| 13/13 [00:03<00:00,  3.70it/s]


Epoch complete: 1/100 (loss: 6.68682986497879)


Epoch 2: 100%|██████████| 13/13 [00:03<00:00,  4.18it/s]


Epoch complete: 2/100 (loss: 2.0822980403900146)


Epoch 3: 100%|██████████| 13/13 [00:03<00:00,  4.16it/s]


...

Epoch complete: 24/100 (loss: 0.08973918331321329)


Epoch 25: 100%|██████████| 13/13 [00:03<00:00,  4.16it/s]


Epoch complete: 25/100 (loss: 0.02192830783315003)


Epoch 26:  77%|███████▋  | 10/13 [00:02<00:00,  3.60it/s]


KeyboardInterrupt: 